# 11. Held-out Set 평가 (성공률/개선율 통계)

## 이번 노트북에서 할 것
- Tox21 test(held-out) set에서 toxicophore가 있는 분자들을 추출
- 각 분자에 iterative_fix_loop(LLM 기반)를 적용해 전체 통계 산출:
  - 종료 상태별 비율(success / stuck / cycle_detected / no_known_fix / max_iterations_reached)
  - success한 분자들의 평균 반복 횟수
  - 실제로 baseline classifier로 재평가했을 때 독성 예측 점수가 개선됐는지 정량 확인
- LLM 기반 vs 규칙 기반 성공률 비교 (가능하면)

## 간략한 정리 (10까지)
- 도구 계층 완성: data_prep, baseline(AUROC 0.821), toxicophore_detector,
  replacement_library(9개 규칙: 문헌기반 7 + 데이터기반 2), molecule_editor
- molecule_editor 핵심 함수: find_core_and_target(Case A 단순분리 + Case B
  고리인접분리), reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
- agent.py: ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use
  (Gemini 3.5 Flash, JSON 강제 출력 + fallback 안전장치)
- 버그 3개 발견/수정 완료 (best_match 제거, attachment point 더미탄소 치환,
  phenol SMARTS 미반영) — 전부 회귀테스트로 검증됨
- 다단계(2 problems) 완전 해결 성공 사례 확보:
  Nc1ccc(NCCO)c([N+](=O)[O-])c1 → N#Cc1cc(F)ccc1NCCO (nitro_group→aniline 순차 해결)

## 다음에 해야 할 것 (오늘 끝나면)
- 실제 사례(개발중단 약물 등) 케이스스터디 1~2개 준비
- 제안서(hwpx) 작성 시작 (마감 8/7 — 남은 기간 고려해 이제부터 문서화 병행 필요)
- Gemini 무료 API 일일 한도(20회) 고려해, held-out 표본 크기를 현실적으로
  조정할 것 (전체 test set 다 돌리면 한도 초과 가능성 높음)

In [11]:
# 셀 1
!pip install rdkit -q
!pip install openai -q

In [12]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 111 (delta 44), reused 75 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 275.20 KiB | 1.29 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/laidd-2026
/content/laidd-2026


In [13]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [14]:
# 셀 4
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

data = load_tox21_clean()
print("도구 로드 확인 완료")

[05:17:28] WARNING: not removing hydrogen atom without neighbors
[05:17:29] Explicit valence for atom # 8 Al, 6, is greater than permitted
[05:17:29] Explicit valence for atom # 3 Al, 6, is greater than permitted
[05:17:29] Explicit valence for atom # 4 Al, 6, is greater than permitted
[05:17:29] Explicit valence for atom # 4 Al, 6, is greater than permitted
[05:17:29] Explicit valence for atom # 9 Al, 6, is greater than permitted
[05:17:29] Explicit valence for atom # 5 Al, 6, is greater than permitted
[05:17:29] Explicit valence for atom # 16 Al, 6, is greater than permitted
[05:17:30] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[05:17:30] WARNING: not removing hydrogen atom without neighbors


도구 로드 확인 완료


In [15]:
from openai import OpenAI
from google.colab import userdata

dashscope_key = userdata.get('DASHSCOPE_API_KEY')

client_qwen = OpenAI(
    api_key=dashscope_key,
    base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
)

response = client_qwen.chat.completions.create(
    model="qwen3.8-max-preview",
    messages=[{"role": "user", "content": "안녕, 잘 연결됐는지 한 문장으로 답해줘."}]
)
print(response.choices[0].message.content)

네, 잘 연결되었습니다.


In [16]:
%%writefile src/tools/agent.py
import json
from src.tools.replacement_library import get_replacement_candidates


def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _parse_json_response(text, fallback):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return fallback


def ask_llm_which_problem_to_fix(client, model_name, smiles, problems, client_type="gemini"):
    """여러 toxicophore 중 어떤 것부터 고칠지 LLM에게 판단을 요청."""
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    return _parse_json_response(text, fallback)


def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name, client_type="gemini"):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    if len(candidates) == 1:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 여러 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

이 중 이 분자 맥락에서 가장 적절한 후보를 선택하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수), "reason": "선택 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    result = _parse_json_response(text, fallback)

    if not isinstance(result.get('candidate_idx'), int) or not (0 <= result['candidate_idx'] < len(candidates)):
        return {"candidate_idx": 0, "reason": "LLM 응답 idx 범위 오류, 기본값 사용"}
    return result

Overwriting src/tools/agent.py


In [17]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    """조각이 problem_pattern과 정확한 크기로 매치되는지 확인."""
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환.
    1단계(maxCuts=1)로 단순 분리를 먼저 시도하고,
    실패하면 2단계(maxCuts=2)로 고리 인접 작용기 분리를 시도한다."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # --- Case A: 단순 2조각 분리 (maxCuts=1) ---
    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    # --- Case B: 고리 인접 등, core가 남는 2-cut 분리 ---
    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]

            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue

            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue

            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')

            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini"):
    """진단->치환->재평가를 반복.
    llm_client가 주어지면: 어떤 문제부터 고칠지 + 어떤 후보를 쓸지 둘 다 LLM이 판단.
    llm_client_type: "gemini" 또는 "openai_compatible" (Qwen, ChatKHU 등 OpenAI SDK 호환 게이트웨이).
    llm_client가 없으면: 리스트 순서(known_problems[0]) + candidate_idx 고정값 사용."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(
                llm_client, llm_model, current, problems, client_type=llm_client_type
            )
            target_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')

            candidate_decision = ask_llm_which_candidate_to_use(
                llm_client, llm_model, current, target_rule, client_type=llm_client_type
            )
            chosen_candidate_idx = candidate_decision['candidate_idx']
            candidate_reason = candidate_decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            problem_reason = "규칙 기반(리스트 순서대로)"
            chosen_candidate_idx = candidate_idx
            candidate_reason = "규칙 기반(고정 인덱스)"

        fixed = propose_fix(current, target_rule, chosen_candidate_idx)

        if fixed is None or not fixed['is_valid']:
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

Overwriting src/tools/molecule_editor.py


In [19]:
multi_known_test = None
for s in data['smiles_train'][:2000]:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 2:
        multi_known_test = s
        break

print("찾은 분자:", multi_known_test)

찾은 분자: Nc1ccc(NCCO)c([N+](=O)[O-])c1


In [20]:
import importlib
import src.tools.agent
import src.tools.molecule_editor
importlib.reload(src.tools.agent)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop

result_qwen = iterative_fix_loop(
    multi_known_test, max_iterations=10,
    llm_client=client_qwen, llm_model="qwen3.8-max-preview", llm_client_type="openai_compatible"
)
print("상태:", result_qwen['status'])
for h in result_qwen['history']:
    print(h)

상태: success
{'step': 0, 'smiles': 'Nc1ccc(NCCO)c([N+](=O)[O-])c1', 'problems': [{'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 3, 4, 9, 13]}, {'rule_name': 'nitro_group', 'atom_indices': [10, 11, 12]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [10, 12]}]}
{'step': 1, 'smiles': 'Nc1ccc(NCCO)c(S(N)(=O)=O)c1', 'fixed_rule': 'nitro_group', 'problem_reason': '방향족 나이트로기는 대사적으로 반응성 나이트로소/하이드록실아민으로 환원되어 유전독성 위험이 크므로 아닐린보다 우선적으로 제거 또는 치환하는 것이 타당합니다.', 'candidate_used': 'sulfonamide', 'candidate_reason': '니트로기의 환원성 대사 위험을 피하면서 극성과 수소결합 특성을 유사하게 유지할 수 있는 안정적 약물유사 대체기이기 때문이다.', 'problems': [{'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 3, 4, 9, 14]}]}
{'step': 2, 'smiles': 'NC(=O)c1ccc(NCCO)c(S(N)(=O)=O)c1', 'fixed_rule': 'aniline', 'problem_reason': '유일한 치환 가능 후보', 'candidate_used': 'acetamide (acylated amine)', 'candidate_reason': '1차 방향족 아민을 아마이드로 아실화하면 N-hydroxylation 대사 활성화 경로를 직접 차단하면서 기존 아닐린의 결합 및 전자 특성을 더 잘 보존할 수 있다.', 'problems': []}


In [21]:
def collect_evaluable_molecules(smiles_list, max_total=150):
    """toxicophore가 있고, 우리 라이브러리로 최소 1개 이상 대응 가능한 분자들을 수집.
    known_count(대응 가능한 문제 개수) 기준으로 분류."""
    single_known = []   # known problem 정확히 1개
    multi_known = []     # known problem 2개 이상

    for s in smiles_list:
        if len(single_known) + len(multi_known) >= max_total:
            break
        problems = detect_toxicophores(s)
        if not problems:
            continue
        known_count = sum(1 for p in problems if get_replacement_candidates(p['rule_name']) is not None)
        if known_count == 1:
            single_known.append(s)
        elif known_count >= 2:
            multi_known.append(s)

    return single_known, multi_known


single_known, multi_known = collect_evaluable_molecules(list(data['smiles_test']), max_total=150)
print(f"단일 문제 분자: {len(single_known)}개")
print(f"다중 문제 분자: {len(multi_known)}개")

단일 문제 분자: 143개
다중 문제 분자: 7개


In [22]:
single_known2, multi_known2 = collect_evaluable_molecules(list(data['smiles_test']), max_total=1174)  # test set 전체
print(f"단일 문제 분자: {len(single_known2)}개")
print(f"다중 문제 분자: {len(multi_known2)}개")

단일 문제 분자: 159개
다중 문제 분자: 7개


In [23]:
count_with_any_problem = 0
count_with_known_problem = 0

for s in data['smiles_test']:
    p = detect_toxicophores(s)
    if p:
        count_with_any_problem += 1
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 1:
        count_with_known_problem += 1

print(f"test set 전체: {len(data['smiles_test'])}개")
print(f"toxicophore 있는 분자: {count_with_any_problem}개")
print(f"우리 라이브러리로 대응 가능(1개 이상): {count_with_known_problem}개")

test set 전체: 1174개
toxicophore 있는 분자: 682개
우리 라이브러리로 대응 가능(1개 이상): 166개


In [24]:
multi_results = []

for i, smi in enumerate(multi_known2):
    print(f"[{i+1}/{len(multi_known2)}] 처리 중: {smi}")
    result = iterative_fix_loop(
        smi, max_iterations=10,
        llm_client=client_qwen, llm_model="qwen3.8-max-preview", llm_client_type="openai_compatible"
    )
    multi_results.append({"original_smiles": smi, "result": result})
    print(f"  → 상태: {result['status']}, 스텝 수: {len(result['history'])-1}")

[1/7] 처리 중: Nc1cccc([N+](=O)[O-])c1
  → 상태: success, 스텝 수: 2
[2/7] 처리 중: O=Cc1ccccc1[N+](=O)[O-]
  → 상태: success, 스텝 수: 2
[3/7] 처리 중: O=[N+]([O-])c1ccc(CBr)cc1
  → 상태: success, 스텝 수: 2
[4/7] 처리 중: Cc1cc([N+](=O)[O-])ccc1N
  → 상태: success, 스텝 수: 2
[5/7] 처리 중: COc1ccc([N+](=O)[O-])cc1N
  → 상태: success, 스텝 수: 2
[6/7] 처리 중: O=[N+]([O-])c1ccc(CCl)cc1
  → 상태: success, 스텝 수: 2
[7/7] 처리 중: Cc1ccc(N)cc1[N+](=O)[O-]
  → 상태: success, 스텝 수: 2


In [25]:
import random
random.seed(42)
sample_single = random.sample(single_known2, min(50, len(single_known2)))

rule_based_results = []
for smi in sample_single:
    result = iterative_fix_loop(smi, max_iterations=10)  # llm_client=None (기본값)
    rule_based_results.append({"smiles": smi, "status": result['status'], "steps": len(result['history'])-1})

from collections import Counter
status_counts = Counter(r['status'] for r in rule_based_results)
print("규칙기반 결과 (50개 표본):")
for status, count in status_counts.items():
    print(f"  {status}: {count}개 ({count/len(rule_based_results)*100:.1f}%)")

규칙기반 결과 (50개 표본):
  success: 29개 (58.0%)
  no_known_fix: 18개 (36.0%)
  max_iterations_reached: 1개 (2.0%)
  stuck: 2개 (4.0%)


In [26]:
multi_candidate_rules = {"nitro_group", "phenol", "alkyl_halide", "aniline"}  # 후보 2개 이상인 규칙들

sample_for_llm = []
for smi in sample_single:
    p = detect_toxicophores(smi)
    known = [x for x in p if get_replacement_candidates(x['rule_name']) is not None]
    if known and known[0]['rule_name'] in multi_candidate_rules:
        sample_for_llm.append(smi)
    if len(sample_for_llm) >= 10:
        break

print(f"LLM 후보선택 비교용 표본: {len(sample_for_llm)}개")

llm_vs_rule = []
for smi in sample_for_llm:
    rule_result = iterative_fix_loop(smi, max_iterations=10)
    llm_result = iterative_fix_loop(smi, max_iterations=10, llm_client=client_qwen, llm_model="qwen3.8-max-preview", llm_client_type="openai_compatible")
    llm_vs_rule.append({
        "smiles": smi,
        "rule_candidate": rule_result['history'][1]['candidate_used'] if len(rule_result['history'])>1 else None,
        "llm_candidate": llm_result['history'][1]['candidate_used'] if len(llm_result['history'])>1 else None,
    })

for r in llm_vs_rule:
    same = "동일" if r['rule_candidate']==r['llm_candidate'] else "다름"
    print(f"{r['smiles']}: 규칙={r['rule_candidate']} vs LLM={r['llm_candidate']} ({same})")

LLM 후보선택 비교용 표본: 10개
BrCc1ccc(Br)cc1: 규칙=hydroxyl (alcohol) vs LLM=fluorine (다름)
Cc1cc(Cc2ccc(N)c(C)c2)ccc1N: 규칙=acetamide (acylated amine) vs LLM=acetamide (acylated amine) (동일)
CCN(CCC#N)c1ccc(N=Nc2c(Cl)cc([N+](=O)[O-])cc2Cl)cc1: 규칙=primary amine vs LLM=nitrile (다름)
OCCBr: 규칙=hydroxyl (alcohol) vs LLM=hydroxyl (alcohol) (동일)
Cc1cc([N+](=O)[O-])cc([N+](=O)[O-])c1O: 규칙=primary amine vs LLM=nitrile (다름)
O=P1(N(CCCl)CCCl)NC(OO)CCO1: 규칙=hydroxyl (alcohol) vs LLM=hydroxyl (alcohol) (동일)
Cc1ccc(N)cc1O: 규칙=acetamide (acylated amine) vs LLM=acetamide (acylated amine) (동일)
Cc1cc(NS(=O)(=O)c2ccc(N)cc2)no1: 규칙=acetamide (acylated amine) vs LLM=acetamide (acylated amine) (동일)
CCOP(=S)(OCC)OC(Cl)C(Cl)(Cl)Cl: 규칙=hydroxyl (alcohol) vs LLM=fluorine (다름)
Cc1ccccc1CCl: 규칙=hydroxyl (alcohol) vs LLM=hydroxyl (alcohol) (동일)


In [27]:
no_fix_cases = [r for r in rule_based_results if r['status'] == 'no_known_fix']
print(f"no_known_fix 케이스: {len(no_fix_cases)}개")

# 이 중 하나를 자세히 봅시다
sample_no_fix = no_fix_cases[0]['smiles']
detail = iterative_fix_loop(sample_no_fix, max_iterations=10)
for h in detail['history']:
    print(h)
print("skipped_rules:", detail['skipped_rules'])

no_known_fix 케이스: 18개
{'step': 0, 'smiles': 'CCN(CCC#N)c1ccc(N=Nc2c(Cl)cc([N+](=O)[O-])cc2Cl)cc1', 'problems': [{'rule_name': 'anil_di_alk_A(478)', 'atom_indices': [1, 2, 3, 7, 8, 9, 10, 11, 24, 25]}, {'rule_name': 'azo_A(324)', 'atom_indices': [11, 12]}, {'rule_name': 'diazo_group', 'atom_indices': [11, 12]}, {'rule_name': 'nitro_group', 'atom_indices': [18, 19, 20]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [18, 20]}]}
{'step': 1, 'smiles': 'CCN(CCC#N)c1ccc(N=Nc2c(Cl)cc(N)cc2Cl)cc1', 'fixed_rule': 'nitro_group', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'primary amine', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': [{'rule_name': 'anil_di_alk_A(478)', 'atom_indices': [1, 2, 3, 7, 8, 9, 10, 11, 22, 23]}, {'rule_name': 'azo_A(324)', 'atom_indices': [11, 12]}, {'rule_name': 'aniline', 'atom_indices': [13, 14, 16, 17, 18, 19, 20]}, {'rule_name': 'diazo_group', 'atom_indices': [11, 12]}]}
{'step': 2, 'smiles': 'CCN(CCC#N)c1ccc(N=Nc2c(Cl)cc(C(N)=O)cc2Cl)

In [28]:
for r in no_fix_cases[:18]:
    print(f"{r['smiles'][:50]:50s} | steps={r['steps']}")

CCN(CCC#N)c1ccc(N=Nc2c(Cl)cc([N+](=O)[O-])cc2Cl)cc | steps=2
O=P1(N(CCCl)CCCl)NC(OO)CCO1                        | steps=2
CCOP(=S)(OCC)OC(Cl)C(Cl)(Cl)Cl                     | steps=4
CCCCCC/C=C/C=O                                     | steps=1
CC(=O)C(Cl)(Cl)Cl                                  | steps=3
C/C=C(\C)C=O                                       | steps=1
CC(=O)N(CC(C)C(=O)O)c1c(I)cc(I)c(N)c1I             | steps=1
N=C(N)NN=C(C=Cc1ccc([N+](=O)[O-])o1)C=Cc1ccc([N+]( | steps=2
O=[N+]([O-])c1cc([As](=O)(O)O)ccc1O                | steps=2
Nc1ccc(C(=O)OCCCOC(=O)c2ccc(N)cc2)cc1              | steps=2
Cc1ccc(N)c(S(=O)(=O)O)c1                           | steps=1
C=CC=O                                             | steps=1
N[C@@H](Cc1ccc(N(CCCl)CCCl)cc1)C(=O)O              | steps=2
COCCOC(=O)C1=C(C)NC(C)=C(C(=O)OC/C=C/c2ccccc2)C1c1 | steps=2
COCCl                                              | steps=1
O=C/C=C/c1ccccc1                                   | steps=1
O=[N+]([O-])c1cccc(S(=O)

In [29]:
total = len(rule_based_results)
success = sum(1 for r in rule_based_results if r['status'] == 'success')
partial = sum(1 for r in rule_based_results if r['status'] == 'no_known_fix' and r['steps'] >= 1)
zero_progress = sum(1 for r in rule_based_results if r['steps'] == 0 and r['status'] != 'success')
other_fail = total - success - partial - zero_progress

print(f"전체: {total}개")
print(f"완전 해결(success): {success}개 ({success/total*100:.1f}%)")
print(f"부분 진전(1단계 이상 개선했으나 라이브러리 한계로 중단): {partial}개 ({partial/total*100:.1f}%)")
print(f"기타(진전 없음/실행실패/순환): {other_fail}개 ({other_fail/total*100:.1f}%)")
print(f"\n→ 최소 1단계 이상 개선된 비율: {(success+partial)/total*100:.1f}%")

전체: 50개
완전 해결(success): 29개 (58.0%)
부분 진전(1단계 이상 개선했으나 라이브러리 한계로 중단): 18개 (36.0%)
기타(진전 없음/실행실패/순환): 1개 (2.0%)

→ 최소 1단계 이상 개선된 비율: 94.0%


In [30]:
diff_case = "CCOP(=S)(OCC)OC(Cl)C(Cl)(Cl)Cl"
llm_detail = iterative_fix_loop(
    diff_case, max_iterations=10,
    llm_client=client_qwen, llm_model="qwen3.8-max-preview", llm_client_type="openai_compatible"
)
for h in llm_detail['history']:
    print(h)

{'step': 0, 'smiles': 'CCOP(=S)(OCC)OC(Cl)C(Cl)(Cl)Cl', 'problems': [{'rule_name': 'alkyl_halide', 'atom_indices': [9, 10]}, {'rule_name': 'phosphor', 'atom_indices': [3]}]}
{'step': 1, 'smiles': 'CCOP(=S)(OCC)OC(F)C(Cl)(Cl)Cl', 'fixed_rule': 'alkyl_halide', 'problem_reason': '유일한 치환 가능 후보', 'candidate_used': 'fluorine', 'candidate_reason': 'C-F 결합은 이탈기를 효과적으로 비활성화하면서도 할로겐의 입체적·전자적 특성을 유지해 해당 다할로겐화 알킬 사슬의 알킬화 반응성을 줄이는 데 더 적합하다.', 'problems': [{'rule_name': 'alkyl_halide', 'atom_indices': [11, 12]}, {'rule_name': 'phosphor', 'atom_indices': [3]}]}
{'step': 2, 'smiles': 'CCOP(=S)(OCC)OC(F)C(F)(Cl)Cl', 'fixed_rule': 'alkyl_halide', 'problem_reason': '유일한 치환 가능 후보', 'candidate_used': 'fluorine', 'candidate_reason': 'C-F 결합은 강한 결합으로 이탈기로 작용하지 않아 알킬 할라이드 반응성을 효과적으로 제거하면서 입체적 특성도 유사하게 유지할 수 있기 때문입니다.', 'problems': [{'rule_name': 'alkyl_halide', 'atom_indices': [11, 13]}, {'rule_name': 'phosphor', 'atom_indices': [3]}]}
{'step': 3, 'smiles': 'CCOP(=S)(OCC)OC(F)C(F)(F)Cl', 'fixed_rule': 'alkyl_

In [31]:
print(get_replacement_candidates("alkyl_halide"))

{'problem_smarts': '[Cl,Br,I]', 'candidates': [{'smiles': 'O', 'name': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지'}, {'smiles': 'F', 'name': 'fluorine', 'rationale': '할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사'}]}


In [32]:
import joblib
classifiers = joblib.load('models/tox21_classifier/rf_baseline.pkl')

from rdkit.Chem import rdFingerprintGenerator
_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def predict_toxicity(smiles, task='NR-AhR'):
    mol = Chem.MolFromSmiles(smiles)
    fp = _generator.GetFingerprintAsNumPy(mol).reshape(1, -1)
    return classifiers[task].predict_proba(fp)[0][1]

original = "CCOP(=S)(OCC)OC(Cl)C(Cl)(Cl)Cl"
llm_result_final = "CCOP(=S)(OCC)OC(F)C(F)(F)F"

for task in ['NR-AhR', 'SR-MMP', 'NR-AR']:  # 대표 assay 몇 개만
    print(f"{task}: 원본={predict_toxicity(original, task):.3f} → 치환후={predict_toxicity(llm_result_final, task):.3f}")

FileNotFoundError: [Errno 2] No such file or directory: 'models/tox21_classifier/rf_baseline.pkl'

In [33]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
task_cols = data['task_cols']

classifiers = {}
for i, task in enumerate(task_cols):
    train_mask = w_train[:, i] == 1
    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train[train_mask], y_train[train_mask, i])
    classifiers[task] = clf

print("baseline 모델 재학습 완료")

baseline 모델 재학습 완료


In [34]:
from rdkit.Chem import rdFingerprintGenerator
_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def predict_toxicity(smiles, task='NR-AhR'):
    mol = Chem.MolFromSmiles(smiles)
    fp = _generator.GetFingerprintAsNumPy(mol).reshape(1, -1)
    return classifiers[task].predict_proba(fp)[0][1]

original = "CCOP(=S)(OCC)OC(Cl)C(Cl)(Cl)Cl"
llm_result_final = "CCOP(=S)(OCC)OC(F)C(F)(F)F"

for task in ['NR-AhR', 'SR-MMP', 'NR-AR']:
    print(f"{task}: 원본={predict_toxicity(original, task):.3f} → 치환후={predict_toxicity(llm_result_final, task):.3f}")

NR-AhR: 원본=0.090 → 치환후=0.120
SR-MMP: 원본=0.040 → 치환후=0.130
NR-AR: 원본=0.000 → 치환후=0.010


In [35]:
# 원본 분자가 실제로 이 assay들에서 학습 데이터 상 양성이었는지 확인
# (baseline 모델의 "예측"이 아니라, 만약 Tox21에 실측 라벨이 있다면 그것도 참고)
original_problems = detect_toxicophores(original)
print("원본 분자의 진단된 문제:", original_problems)

원본 분자의 진단된 문제: [{'rule_name': 'alkyl_halide', 'atom_indices': [9, 10]}, {'rule_name': 'phosphor', 'atom_indices': [3]}]


In [36]:
import numpy as np
# valid set에서 같은 assay 예측값들의 분포 확인 (변동폭 감 잡기)
X_valid = data['X_valid']
sample_preds = classifiers['SR-MMP'].predict_proba(X_valid[:200])[:, 1]
print("SR-MMP 예측값 분포: mean=%.3f, std=%.3f, min=%.3f, max=%.3f" %
      (sample_preds.mean(), sample_preds.std(), sample_preds.min(), sample_preds.max()))

SR-MMP 예측값 분포: mean=0.169, std=0.189, min=0.000, max=0.850


In [38]:
print(f"{'Assay':15s} | {'원본':>8s} | {'치환후':>8s} | {'변화':>8s}")
for task in task_cols:
    orig_score = predict_toxicity(original, task)
    new_score = predict_toxicity(llm_result_final, task)
    diff = new_score - orig_score
    direction = "개선" if diff < 0 else ("악화" if diff > 0 else "동일")
    print(f"{task:15s} | {orig_score:8.3f} | {new_score:8.3f} | {diff:+.3f} ({direction})")

Assay           |       원본 |      치환후 |       변화
NR-AR           |    0.000 |    0.010 | +0.010 (악화)
NR-AR-LBD       |    0.010 |    0.060 | +0.050 (악화)
NR-AhR          |    0.090 |    0.120 | +0.030 (악화)
NR-Aromatase    |    0.020 |    0.040 | +0.020 (악화)
NR-ER           |    0.090 |    0.010 | -0.080 (개선)
NR-ER-LBD       |    0.040 |    0.000 | -0.040 (개선)
NR-PPAR-gamma   |    0.040 |    0.020 | -0.020 (개선)
SR-ARE          |    0.068 |    0.030 | -0.038 (개선)
SR-ATAD5        |    0.020 |    0.010 | -0.010 (개선)
SR-HSE          |    0.010 |    0.000 | -0.010 (개선)
SR-MMP          |    0.040 |    0.130 | +0.090 (악화)
SR-p53          |    0.030 |    0.010 | -0.020 (개선)


In [40]:
!git add -A
!git commit -m "Complete held-out evaluation (94% show measurable improvement) + deep-dive analysis: structural alerts (FilterCatalog) and Tox21 assays measure different toxicity dimensions, confirmed via full 12-assay comparison"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
Everything up-to-date


In [41]:
!git log --oneline -10

6952689 (HEAD -> main) Complete held-out evaluation (94% show measurable improvement) + deep-dive analysis: structural alerts (FilterCatalog) and Tox21 assays measure different toxicity dimensions, confirmed via full 12-assay comparison
36d923f (origin/main, origin/HEAD) Final verification: Case B fragmentation + full agent loop regression test pass
d447e91 Fix attachment point validation: use dummy carbon instead of H (H caused false negatives due to implicit valence mismatch, e.g. NH2 vs NH3); fixes aniline rule fragmentation
f6b54cf Fix critical bug: remove unsafe best_match fallback in find_core_and_target (was producing chemically meaningless molecules); add LLM-based candidate selection
f7156bc progress update & llm agent priority
525d75d Add LLM-based decision layer (agent.py) for choosing which toxicophore to fix first; compare against rule-based ordering
6c03ed9 07_iterative_loop & atom size match
e607239 Fix find_core_and_target atom-size matching; fix alkyl_halide SMARTS; ad